# Kaggle Final Submission

**Due:** see Canvas. **Rubric:** the *Final Submission* section of `Course_Documents/Grading_Rubrics.md`.

This notebook is a skeleton for the final submission. It picks up where Checkpoint 1 left off: you'll compare several model families, tune one setting, engineer at least one feature, and make your final submission, with cross-validation evidence for every choice. It gives you the structure and the mechanical parts: loading the data, a function that compares models, the tuning plot, and writing the file Kaggle expects. The decisions, and the reasons for them, are yours.

Your main reference is *Final Submission Guidance* (`final_guidance.ipynb`, in this folder). It works through every step below on the Adult census data, and its Section 1 has a complete, ready-to-copy pipeline for each of five model families. Sections 2–4 below name the part of it to reread. (It downloads its data the first time it runs, so it needs an internet connection.)

### Before you start

1. Copy this `Final/` folder into your own project folder, next to your `Checkpoint_1/` folder, and work in the copy. As before, don't edit the files in the original `Project/`.
2. Add a `data/` folder next to this notebook, and put `train.csv` and `test.csv` in it: the same two files as Checkpoint 1.
3. Copy your `log.md` from `Checkpoint_1/` into this folder, and keep adding a row for every Kaggle submission.

   ```
   Project_YourName/
   ├── Checkpoint_1/
   │   └── ...
   └── Final/
       ├── final.ipynb             (this notebook)
       ├── final_guidance.ipynb    (the reference notebook)
       ├── log.md                  (copied from Checkpoint_1/, and kept current)
       └── data/
           ├── train.csv
           └── test.csv
   ```

   On Colab, make a folder named `data` in the Files sidebar and upload both CSV files into it.

### How this notebook works

- Code cells that say `# YOUR CODE HERE` are yours. Add more cells wherever you need them.
- Cells that say *YOUR ANSWER HERE* are yours. Write in full sentences.
- Every other cell is provided. Read it, and run it as it is.
- Restart & Run All should finish in a few minutes. If one of your cells takes much longer, try fewer values or a smaller model.

### What you'll hand in

| Label | Type | Task |
| --- | --- | --- |
| 1(a) | Code | Your Checkpoint 1 column rearranging, revised |
| 1(b) | Code | Your Checkpoint 1 preprocessing, revised |
| 1(c) | Written | What you changed since Checkpoint 1 |
| 2(a) | Code | Compare three or more model families |
| 2(b) | Written | Is your best model's lead real? |
| 3(a) | Code | Tune one setting |
| 3(b) | Written | Which value, and why? |
| 4(a) | Code | Build a new feature |
| 4(b) | Code | Compare with and without it |
| 4(c) | Written | Did it help, and why? |
| 5(a) | Code | Fit the final model and write `submission.csv` |
| 5(b) | Written | CV accuracy and leaderboard score |
| 6(a) | Written | Summary |
| 6(b) | Written | CV against the leaderboard |
| 6(c) | Written | What didn't work |
| 6(d) | Written | AI use |

Submit `final.ipynb`, `log.md`, and any `.py` helper files it imports in Canvas. Don't submit the data files.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# The five model families in the guidance's menu
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

SEED = 463    # pass random_state=SEED everywhere it's accepted, so your numbers are reproducible

DATA = Path('data')
for name in ['train.csv', 'test.csv']:
    if not (DATA / name).exists():
        raise FileNotFoundError(f"Can't find {DATA / name}. Put train.csv and test.csv in a folder "
                                "named 'data' next to this notebook (see 'Before you start').")

train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
print(f'train: {train.shape[0]:,} rows, {train.shape[1]} columns')
print(f'test:  {test.shape[0]:,} rows, {test.shape[1]} columns')

**Provided (below).** The same setup as Checkpoint 1. `y` is the target. `X` holds every other column of the training data, and `X_test` holds the same columns for the test rows. `test_ids` keeps the test passengers' IDs in order, for the file you'll upload.

The same rule holds for the whole notebook: anything you do to `X` (splitting a column, adding a column, dropping one), do to `X_test` too, in the same way.

In [ ]:
y = train['Transported']
X = train.drop(columns=['Transported'])
X_test = test.copy()
test_ids = test['PassengerId']
X.head()

## 1 · From Checkpoint 1

Start from your Checkpoint 1 notebook and the feedback on it. Fix anything the feedback pointed out as you copy your code in.

**1(a) Code.** Paste your Checkpoint 1 **2(a)** code: the splitting and rearranging you did to both `X` and `X_test`, revised using your feedback.

In [ ]:
# 1(a)
# YOUR CODE HERE


**1(b) Code.** Paste your Checkpoint 1 **2(d)** preprocessing, revised using your feedback.

Keep your columns in two named lists, `numeric_columns` and `categorical_columns`, and name the whole step `preprocess`, as in Checkpoint 1. Section 4 builds a second version of `preprocess` by adding your new feature to one of the lists. (If you named the per-type pipelines `numeric_pipeline` and `categorical_pipeline`, as the guidance does, Section 4's example code fits your notebook with almost no changes.)

In [ ]:
# 1(b)
# YOUR CODE HERE


**1(c) Written.** What did you change from your Checkpoint 1 code, and why? One sentence per change is enough. If the feedback asked for no changes, say so.

**Answer:**
_YOUR ANSWER HERE_


## 2 · Compare model families

Reread *Final Submission Guidance*, Sections 1 and 2.

**Provided.** `folds` is the same 5-fold split as Checkpoint 1. Use it for every cross-validation in this notebook, so every model is scored on exactly the same validation rows.

`compare` is the function from Guidance Section 2. It cross-validates each model in a dictionary and returns one row per model: the mean and standard deviation of its 5 fold accuracies, each fold's accuracy, all in percent, and how long it took.

In [ ]:
folds = KFold(n_splits=5, shuffle=True, random_state=SEED)

def compare(models, X, y, folds):
    # Cross-validate each model on the same folds; return one row per model.
    rows = []
    for name, model in models.items():
        start = time.perf_counter()
        scores = cross_val_score(model, X, y, cv=folds)
        row = {'model': name,
               'CV accuracy (%)': 100 * scores.mean(),
               'std (%)': 100 * scores.std()}
        for k, score in enumerate(scores, start=1):
            row[f'fold {k}'] = 100 * score       # the same 5 folds for every model
        row['seconds'] = time.perf_counter() - start
        rows.append(row)
    return pd.DataFrame(rows).set_index('model').round(2)

**2(a) Code.** Make a dictionary named `models` that holds your Checkpoint 1 model and **at least two more model families**, at least one of them a tree ensemble (a random forest or gradient boosting). Every pipeline uses your `preprocess`. Pass `random_state=SEED` to any model that accepts it. Then run `compare(models, X, y, folds)` and display the table.

<details>
<summary>Hint</summary>

Guidance Section 1 is a menu: each family has a complete pipeline you can copy, with notes on the settings worth changing. Swap in your own `preprocess`, and use `random_state=SEED` in place of `random_state=0`. Guidance Section 2 shows the dictionary and the call to `compare`.

</details>

In [ ]:
# 2(a)
# YOUR CODE HERE


**2(b) Written.** Is your best model's lead over the runner-up real? Use the three questions from Guidance Section 2:

1. How big is the gap, compared with the spread (the standard deviations)?
2. Does the better model win in every fold?
3. Is the gap big enough to matter? The test set has 2,174 passengers, so a gap of 1 percentage point is a passenger count of about 22.

**Answer:**
_YOUR ANSWER HERE_


## 3 · Tune one setting

Reread *Final Submission Guidance*, Section 3.

**3(a) Code.** Choose one model worth improving and one of its settings from the guidance menu. Cross-validate 5–7 widely spread values of that setting with `compare`, and name the resulting table `tuning_table`. Also set `setting` to the setting's name, as a string, such as `'n_neighbors'`. The plot below uses both names.

<details>
<summary>Hint</summary>

Guidance Section 3 does this for the random forest's `min_samples_leaf`. Its dictionary comprehension builds one pipeline per value, with the value as the key, so the values become the table's row labels and the plot's x-axis. Your values should be numbers.

</details>

In [ ]:
# 3(a)
# YOUR CODE HERE


**Provided.** The plot of your tuning results: each mean, with a bar one standard deviation above and below it. It reads `setting` and `tuning_table` from 3(a), so it raises a `NameError` until you've run that cell.

The x-axis is logarithmic, which spaces values that grow by a factor evenly. If your values are evenly spaced instead (such as 10, 20, 30, 40), delete the `set_xscale` line.

In [ ]:
values = tuning_table.index
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(values, tuning_table['CV accuracy (%)'], yerr=tuning_table['std (%)'], marker='o', capsize=4)
ax.set_xscale('log')                          # delete this line if your values are evenly spaced
ax.set_xticks(values, labels=values)          # label the values you tried
ax.set_title(f'CV accuracy against {setting}')
ax.set_xlabel(setting)
ax.set_ylabel('CV accuracy (%), mean ± std')
plt.show()

**3(b) Written.** Which value do you choose, and why? Use the three questions on the best value and its closest competitor. When the gap is too small to matter, prefer the simpler model.

**Answer:**
_YOUR ANSWER HERE_


## 4 · Engineer a feature

Reread *Final Submission Guidance*, Section 4, including *The rule for where a feature goes*.

You need **at least one** engineered feature. Evaluate with and without it on the same model and folds. A feature that doesn't help still counts for this assignment, as long as you evaluate it and explain the result. Math on a numeric column, like squaring or a log, is optional: a feature built from a string or from related rows fully meets the requirement.

**4(a) Code.** Build your new feature(s) as new columns in copies of `X` and `X_test`, named `X_more_features` and `X_test_more_features`. Build only features that use a row's own values, or `X` and `X_test` without the label, since this cell runs before cross-validation.

<details>
<summary>Hint 1: where to look</summary>

Reread the sources table at the start of Guidance Section 4. Then look at the columns your Checkpoint 1 model leaves out. Is there a usable piece of information inside any of them? The end of Guidance Section 4 also suggests one place to look.

</details>

<details>
<summary>Hint 2: the code pattern</summary>

Step 1 of the guidance's feature example: copy `X`, add one column per feature, and do the same to `X_test`. If you build several features, the guidance's optional `add_features` function keeps `X_more_features` and `X_test_more_features` in step.

</details>

In [ ]:
# 4(a)
# YOUR CODE HERE


**4(b) Code.** Build a second preprocessing step that also lists your new column(s), and use `compare` to cross-validate one model with and without the feature, on `X_more_features`. Use the model you chose in Section 3, with the setting you chose. Name the resulting table `feature_table`.

<details>
<summary>Hint</summary>

Steps 2 and 3 of the guidance's feature example. Don't forget to add the new column's name to a list: a column that isn't in either list is silently dropped. Both versions can run on `X_more_features`, because the original `preprocess` ignores columns it isn't given.

</details>

In [ ]:
# 4(b)
# YOUR CODE HERE


**4(c) Written.** Did your feature help? Use the three questions. Why do you think it helped this model, or didn't? Would you expect the same result for a different model family?

**Answer:**
_YOUR ANSWER HERE_


## 5 · Final model and submission

**Provided.** `write_kaggle_file` is the same function as Checkpoint 1. It saves your test-set predictions in the format the competition expects, and it stops with a message if the predictions have the wrong length or look like probabilities.

In [ ]:
def write_kaggle_file(predictions, path='submission.csv'):
    # Save test-set predictions as the CSV file you upload to Kaggle.
    predictions = np.asarray(predictions)
    if len(predictions) != len(test_ids):
        raise ValueError(f'Expected {len(test_ids):,} predictions, one per row of test.csv; '
                         f'got {len(predictions):,}.')
    if not set(np.unique(predictions)) <= {0, 1}:
        raise ValueError('Predictions must be True/False or 1/0. '
                         'Did you pass probabilities from predict_proba instead of predict?')
    file = pd.DataFrame({'PassengerId': test_ids, 'Transported': predictions.astype(bool)})
    file.to_csv(path, index=False)
    print(f'Wrote {path}: {len(file):,} rows, {file["Transported"].mean():.1%} predicted True')

**5(a) Code.** Choose your final pipeline from Sections 2–4. Fit it on **all** of the training rows, predict for the test rows, and call `write_kaggle_file`. If your final pipeline uses your new feature, fit on `X_more_features` and predict for `X_test_more_features`. Then upload the file on the competition's **Submit Predictions** page, and select it for final scoring.

This is the submission your notebook must reproduce: Restart & Run All should write exactly the file you selected.

In [ ]:
# 5(a)
# YOUR CODE HERE


**5(b) Written.** What is your final model's CV accuracy (mean ± standard deviation), and what was its public leaderboard score?

**Answer:**
_YOUR ANSWER HERE_


## 6 · Write-up

Short answers, in full sentences. Each one should stand on its own for a reader who hasn't seen the code.

Before you answer, update `log.md` with a row for every submission you made.

**6(a) Written.** **Summary.** In one paragraph: the problem, your final model and its settings, the features it uses, and its CV accuracy. Why did you choose it over the runner-up? Cite your CV evidence, not your leaderboard rank.

**Answer:**
_YOUR ANSWER HERE_


**6(b) Written.** **CV against the leaderboard.** Compare your final model's CV accuracy with its public leaderboard score. Is the gap bigger than your CV standard deviation? Two things to keep in mind: the public leaderboard scores only 30% of the test rows (652 passengers), and a CV score you chose because it was the best of several is a little optimistic (Guidance Section 3).

**Answer:**
_YOUR ANSWER HERE_


**6(c) Written.** **What didn't work.** Describe at least one thing you tried that didn't help (a model, a setting, or a feature), and what it taught you.

**Answer:**
_YOUR ANSWER HERE_


**6(d) Written.** **AI use.** Which AI tools did you use on this project, if any, and what for?

**Answer:**
_YOUR ANSWER HERE_


## Before you submit

1. **Kernel → Restart & Run All.** Check that every cell ran and every output and answer is visible.
2. Check that the `submission.csv` it writes is the submission you selected on Kaggle.
3. Check that the notebook loads the data from `data/` with the relative paths above, not a path that only exists on your computer.
4. Check that `log.md` has a row for every Kaggle submission.
5. Submit `final.ipynb`, `log.md`, and any `.py` helper files it imports in Canvas. Don't submit the data files.